# Day 4: S2 and the Hilbert curve (S2 vs H3 on the same polygon)

**Theory:** 3Blue1Brown the Hilbert curve · [S2 cell hierarchy](https://s2geometry.io/devguide/s2cell_hierarchy.html) · [Perone: S2](https://blog.christianperone.com/2015/08/googles-s2-geometry-on-the-sphere-cells-and-hilbert-curve/)

**What we gonna do:** where does S2 used efficiently than H3 and otherwise.

| | S2 | H3 |
|---|---|---|
| format | square(cube projection) | hexagon (+12 pentagons) |
| area inside | **not equal** (spread ~2x) | almost equal |
| hierarchy | **strict**: parent = prefix id | approximate, child do not add up to parent |
| id order | **Hilbert curve** → neighbouring id ≈ neighbouring on map | no order |
| neighbour | 4 edges (+ corners), unequal distances | 6 neighbours, equal distances |

**Hypothesis which gonna be checked:** S2 decomposes into a small number of *contiguous ID ranges* (a range scan on a B-tree), whereas H3 decomposes only into a list of keys

In [1]:
import math, os, json
import numpy as np, pandas as pd
import h3, s2sphere as s2
from shapely.geometry import Polygon, mapping, shape
from shapely.ops import unary_union
from pyproj import Geod

OFFICE = (43.2380, 76.9450)        # Almaty, same in d3
R_KM = 6371.0088                   # earth avg radius
GEOD = Geod(ellps="WGS84")
os.makedirs("m01", exist_ok=True)
print(h3.__version__)

4.5.0


## Step 1 — same polygon for all comparison

Take one administrative district (Almaty) with OSMnx and
cache it to `data/processed/` so the notebook won't depend on network. If the geocoder is unavailable, use a proper fallback:

a square of approximately 10x10 km centered on the office (indicate in the output which option was used).

**Geocoder trap:** on `"Bostandyk District, Almaty, Kazakhstan"` Nominatim returns a building of 0.0002 km², but on `"Almaty, Kazakhstan"` - Almaty **Region** (104 806 km²). Hence, checking the area to ensure it falls within a reasonable range: if an incorrect polygon is inadvertently selected, it will invalidate all subsequent figures.

In [2]:
POLY_CACHE = "../../data/processed/m01_polygon.geojson"
PLACE = "Бостандыкский район, Алматы, Казахстан"   # "Bostandyk District"  
SANE_KM2 = (10, 400)                               # "Almaty, Kazakhstan"

def load_polygon():
    if os.path.exists(POLY_CACHE):
        with open(POLY_CACHE, encoding="utf-8") as f:
            gj = json.load(f)
        return shape(gj["geometry"]), gj["properties"]["source"]
    try:
        import osmnx as ox
        gdf = ox.geocode_to_gdf(PLACE)
        poly = gdf.geometry.iloc[0]
        km2 = abs(GEOD.geometry_area_perimeter(poly)[0]) / 1e6
        if not (SANE_KM2[0] <= km2 <= SANE_KM2[1]):    # геокодер попал не туда
            raise ValueError(f"{PLACE}: {km2:.2f} km2 вне {SANE_KM2}")
        src = f"OSM: {PLACE}"
    except Exception as e:                      # нет сети / геокодер молчит / не тот объект
        print("geocode failed:", type(e).__name__, e)
        dlat, dlon = 0.045, 0.062               # ~10 x 10 км
        poly = Polygon([(OFFICE[1]-dlon, OFFICE[0]-dlat), (OFFICE[1]+dlon, OFFICE[0]-dlat),
                        (OFFICE[1]+dlon, OFFICE[0]+dlat), (OFFICE[1]-dlon, OFFICE[0]+dlat)])
        src = "fallback square ~10x10 km around OFFICE"
    os.makedirs(os.path.dirname(POLY_CACHE), exist_ok=True)
    with open(POLY_CACHE, "w", encoding="utf-8") as f:
        json.dump({"type": "Feature", "properties": {"source": src},
                   "geometry": mapping(poly)}, f)
    return poly, src

POLY, POLY_SRC = load_polygon()
POLY_KM2 = abs(GEOD.geometry_area_perimeter(POLY)[0]) / 1e6
print(POLY_SRC)
print(f"Area: {POLY_KM2:.2f} km², vertices: {len(POLY.exterior.coords)}")

OSM: Бостандыкский район, Алматы, Казахстан
Area: 98.55 km², vertices: 1069


## Step 2 - H3 covering

Two modes; the distinction is key to answering "why a final re-rank is necessary after a ring search":

* `h3shape_to_cells` - a cell is included if **its center** lies within the polygon (edges may remain uncovered);
* `h3shape_to_cells_experimental(..., contain="overlap")` - any intersection is included (ensuring full coverage, with some margin).

In [3]:
def cover_h3(poly, res, mode="center"):
    shp = h3.geo_to_h3shape(mapping(poly))
    if mode == "center":
        return list(h3.h3shape_to_cells(shp, res))
    return list(h3.h3shape_to_cells_experimental(shp, res, contain="overlap"))

h3_cells = {(res, mode): cover_h3(POLY, res, mode)
            for res in (7, 8, 9) for mode in ("center", "overlap")}

for k, v in h3_cells.items():
    print(k, len(v))

assert len(h3_cells[(8, "overlap")]) >= len(h3_cells[(8, "center")])
assert len(h3_cells[(9, "center")]) > len(h3_cells[(8, "center")]) > len(h3_cells[(7, "center")])

(7, 'center') 15
(7, 'overlap') 35
(8, 'center') 111
(8, 'overlap') 164
(9, 'center') 786
(9, 'overlap') 919


## Step 3 - S2 coverage (and its trap)

**In `s2sphere` no `S2Polygon`/`S2Loop`** - `RegionCoverer` has only `Cap` in regions and
`LatLngRect`. Thus, we do coverage in 2 ways:

1. `RegionCoverer.get_covering(LatLngRect)` with the bbox - returns a **multi-level** covering
   (small cells on edges, big ones in center), limited `max_cells`;
2. remove cells that do not intersect the polygon itself (shapely).

`max_cells` - is not about «how much you want» but a budget: coverer decides itself where to divide. Use
`min_level`/`max_level`/`max_cells` and see how over-coverage changes.

**Read `coverage` of S2 with correction:** We cover the bbox and then discard the cells that do not intersect the polygon; therefore ~2x overhead is a limitation of `s2sphere`, **not** an inherent property of S2.
A proper `S2RegionCoverer` implementation for an `S2Polygon` (in C++/Go, or via `s2geometry` using pybind) would subdivide the boundary down to `max_level`, yielding a coverage comparable to H3’s overlap. This must be stated transparently in the conclusions; otherwise, it would amount to a biased comparison favoring H3.

In [5]:
def s2_cell_polygon(cid: s2.CellId) -> Polygon:
    c = s2.Cell(cid)
    pts = []
    for i in range(4):
        ll = s2.LatLng.from_point(c.get_vertex(i))
        pts.append((ll.lng().degrees, ll.lat().degrees))
    return Polygon(pts)

def s2_cell_area_km2(cid: s2.CellId) -> float:
    return s2.Cell(cid).exact_area() * R_KM * R_KM      # steradians -> km2

def cover_s2(poly, min_level, max_level, max_cells, clip=True):
    minx, miny, maxx, maxy = poly.bounds
    rect = s2.LatLngRect(s2.LatLng.from_degrees(miny, minx), s2.LatLng.from_degrees(maxy, maxx))
    rc = s2.RegionCoverer()
    rc.min_level, rc.max_level, rc.max_cells = min_level, max_level, max_cells
    ids = list(rc.get_covering(rect))
    if clip:
        ids = [i for i in ids if s2_cell_polygon(i).intersects(poly)]
    return ids

s2_cells = {("s2", mc): cover_s2(POLY, 10, 14, mc) for mc in (100, 500, 2000)}
for (_, mc), ids in s2_cells.items():
    print("max_cells", mc, "->", len(ids), "cells, levels", sorted({i.level() for i in ids}))

# уровень S2 ~ res H3: L13 ≈ res 8 по площади — проверь по таблице ниже
assert all(10 <= i.level() <= 14 for ids in s2_cells.values() for i in ids)

max_cells 100 -> 39 cells, levels [11, 12, 13, 14]
max_cells 500 -> 45 cells, levels [11, 12, 13, 14]
max_cells 2000 -> 45 cells, levels [11, 12, 13, 14]


## Step 4 - comparison table

Metrics:

* `n_cells`- how many keys are needed to describe a district;
* `coverage` - area of ​​merged cells / polygon area (>1 — detects excess, <1 — results in loss of edges);
* `inside_frac` — what share of the polygon itself is actually covered;
* `area_min/med/max`, `area_spread` — spread of cell area within one covering (this is where S2 is uneven);
* `levels` — how many distinct levels/resolutions are mixed in one covering.

In [6]:
def stats(name, geoms, areas_km2, levels):
    merged = unary_union(geoms)
    cov_km2 = abs(GEOD.geometry_area_perimeter(merged)[0]) / 1e6
    inter = abs(GEOD.geometry_area_perimeter(merged.intersection(POLY))[0]) / 1e6
    a = np.array(areas_km2)
    return {"scheme": name, "n_cells": len(geoms),
            "coverage": cov_km2 / POLY_KM2,              # >1 = запас, <1 = недобор
            "inside_frac": inter / POLY_KM2,
            "area_min": a.min(), "area_med": float(np.median(a)), "area_max": a.max(),
            "area_spread": a.max() / a.min(), "levels": len(set(levels))}

rows = []
for (res, mode), cells in h3_cells.items():
    rows.append(stats(f"H3 res{res} ({mode})",
                      [Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in cells],
                      [h3.cell_area(c, unit="km^2") for c in cells],
                      [res] * len(cells)))
for (_, mc), ids in s2_cells.items():
    rows.append(stats(f"S2 L10-14 (max_cells={mc})",
                      [s2_cell_polygon(i) for i in ids],
                      [s2_cell_area_km2(i) for i in ids],
                      [i.level() for i in ids]))

cmp = pd.DataFrame(rows).set_index("scheme").round(3)
cmp

,n_cells,coverage,inside_frac,area_min,area_med,area_max,area_spread,levels
scheme,,,,,,,,
H3 res7 (center),15,0.940,0.776,6.159,6.162,6.164,1.001,1
H3 res7 (overlap),35,2.192,1.000,6.159,6.162,6.164,1.001,1
H3 res8 (center),111,0.993,0.910,0.880,0.880,0.881,1.001,1
H3 res8 (overlap),164,1.468,1.000,0.880,0.880,0.881,1.001,1
H3 res9 (center),786,1.005,0.968,0.126,0.126,0.126,1.001,1
H3 res9 (overlap),919,1.175,1.000,0.126,0.126,0.126,1.001,1
S2 L10-14 (max_cells=100),39,1.980,1.000,0.247,0.995,15.907,64.424,4
S2 L10-14 (max_cells=500),45,1.935,1.000,0.247,0.994,15.907,64.431,4
S2 L10-14 (max_cells=2000),45,1.935,1.000,0.247,0.994,15.907,64.431,4


In [7]:
# Проверки к таблице (подправь порог, если взял другой полигон):
# 1) у H3 внутри одного res разброс площадей почти нулевой
assert cmp.loc["H3 res8 (center)", "area_spread"] < 1.1
# 2) у S2 в разноуровневом покрытии разброс кратный
assert cmp.loc["S2 L10-14 (max_cells=500)", "area_spread"] > 2
# 3) режим center недобирает край, overlap — перебирает
assert cmp.loc["H3 res8 (center)", "inside_frac"] < 1.0 <= cmp.loc["H3 res8 (overlap)", "coverage"]
print("ok")

ok


## Шаг 5 — главный аргумент S2: локальность по кривой Гильберта

`CellId` — это int64, позиция на кривой Гильберта; `range_min()/range_max()` дают интервал
листовых id, покрытых ячейкой. Если покрытие укладывается в K непрерывных интервалов, то
«всё в этом районе» = **K range-запросов `WHERE s2_id BETWEEN a AND b`** по обычному B-tree.

Деталь, на которой легко сломаться: у листовых ячеек младший бит — маркер, поэтому соседние
листовые id различаются на **2**, а не на 1 (отсюда `step=2`).

In [8]:
def hilbert_runs(cell_ids, step=2):
    iv = sorted((c.range_min().id(), c.range_max().id()) for c in cell_ids)
    runs, cur_end = 1, iv[0][1]
    for lo, hi in iv[1:]:
        if lo > cur_end + step:
            runs += 1
        cur_end = max(cur_end, hi)
    return runs

ids = s2_cells[("s2", 500)]
union = s2.CellUnion(list(ids)); union.normalize()
runs = hilbert_runs(list(union.cell_ids()))
print(f"S2: {len(ids)} ячеек -> {union.num_cells()} после normalize -> {runs} диапазонов id")
print("H3: ключей", len(h3_cells[(8, 'center')]), "- диапазонов нет, порядок id не географический")
assert runs < union.num_cells()

S2: 45 ячеек -> 45 после normalize -> 19 диапазонов id
H3: ключей 111 - диапазонов нет, порядок id не географический


**Задание 1.** Докажи, что у H3 географического порядка нет: отсортируй `h3_cells[(8,"center")]`
по `int(cell, 16)` и посчитай среднее `h3.grid_distance` между подряд идущими id. Сравни с тем же
для S2 (подряд идущие `CellId` → расстояние между центрами в метрах, `GEOD.inv`). Одна цифра в вывод.

**Задание 2.** Строгая иерархия S2: возьми ячейку L13, её `parent(10)`, проверь
`parent.contains(child)` и что сумма площадей всех детей L13 внутри родителя равна площади
родителя. Повтори для H3 (`cell_to_parent` / `cell_to_children`) — и объясни в двух строках,
почему у H3 так не выходит и чем это грозит при агрегации спроса по уровням.

**Задание 3.** Убери bbox-фору: напиши свой coverer по полигону — начни с ячеек L10,
каждую, которая пересекает границу, дроби через `cell_id.children()` до `max_level`, внутренние
оставляй как есть. Сравни `coverage` с таблицей выше: насколько `s2sphere` переплачивает за
отсутствие `S2Polygon`?

**Задание 4 (на выбор).** Geofencing: сколько ячеек нужно, чтобы описать зону обслуживания с
точностью ±5% по площади, у S2 и у H3? Это и есть ответ на self-check
«Pick H3, S2 or geohash for (a) surge heatmap, (b) geofencing, (c) nearest driver».

In [ ]:
# Задание 1
# 1. Сортируем ячейки по целочисленному представлению индекса
h3_sorted = sorted(h3_cells[(8, "center")], key=lambda c: int(c, 16))

def h3_dist_m(a, b):
    (lat1, lon1), (lat2, lon2) = h3.cell_to_latlng(a), h3.cell_to_latlng(b)
    return GEOD.inv(lon1, lat1, lon2, lat2)[2]           # в GEOD.inv сначала долгота!

# 2. Шаги по сетке и метры между соседними в списке
h3_steps, h3_dists, h3_skipped = [], [], 0
for a, b in zip(h3_sorted[:-1], h3_sorted[1:]):
    h3_dists.append(h3_dist_m(a, b))
    try:
        h3_steps.append(h3.grid_distance(a, b))
    except h3.H3BaseException:
        # grid_distance падает через пентагон / на стыке граней икосаэдра: считаем, а не глотаем молча
        h3_skipped += 1

h3_steps, h3_dists = np.array(h3_steps), np.array(h3_dists)
print("=== H3 (res 8) ===")
print(f"шаги:  среднее {h3_steps.mean():.2f}, медиана {np.median(h3_steps):.0f}, "
      f"максимум {h3_steps.max()}, пропущено пар {h3_skipped}")
print(f"метры: среднее {h3_dists.mean():.0f}, медиана {np.median(h3_dists):.0f}, максимум {h3_dists.max():.0f}")

In [ ]:
# 1. Одноуровневое покрытие S2 (L13 ≈ H3 res8: у обеих сеток шаг до соседа ~1 км) и сортировка по 64-битному id
s2_l13 = cover_s2(POLY, 13, 13, 100000)
s2_sorted = sorted(s2_l13, key=lambda c: c.id())

def s2_dist_m(a, b):
    A, B = a.to_lat_lng(), b.to_lat_lng()
    return GEOD.inv(A.lng().degrees, A.lat().degrees, B.lng().degrees, B.lat().degrees)[2]

# 2. Геодезическое расстояние между центрами подряд идущих CellId
s2_dists = np.array([s2_dist_m(a, b) for a, b in zip(s2_sorted[:-1], s2_sorted[1:])])
print("=== S2 (level 13) ===")
print(f"метры: среднее {s2_dists.mean():.0f}, медиана {np.median(s2_dists):.0f}, максимум {s2_dists.max():.0f}")
# максимум ~18 км не баг: кривая выходит из полигона и возвращается в другом месте, а clip выкинул ячейки снаружи

In [ ]:
# 3. Общая шкала и база для сравнения
# шаг до соседа по ребру: у H3 res8 и S2 L13 он ~1 км, поэтому метры сравнимы напрямую
h3_step_m = np.mean([h3_dist_m(h3_sorted[0], n) for n in h3.grid_ring(h3_sorted[0], 1)])
s2_step_m = np.mean([s2_dist_m(s2_sorted[0], n) for n in s2_sorted[0].get_edge_neighbors()])

# случайный порядок тех же ячеек = «порядка нет совсем»
rng = np.random.default_rng(0)
def random_mean_m(cells, dist, n=50):
    out = []
    for _ in range(n):
        p = [cells[i] for i in rng.permutation(len(cells))]
        out.append(np.mean([dist(a, b) for a, b in zip(p[:-1], p[1:])]))
    return float(np.mean(out))

NEAR_M = 1500   # «рядом» = соседняя ячейка (шаг ~1 км, диагональ у S2 ~1.4 км)
order = pd.DataFrame({
    "cells":         [len(h3_sorted), len(s2_sorted)],
    "step_m":        [h3_step_m, s2_step_m],
    "mean_m":        [h3_dists.mean(), s2_dists.mean()],
    "median_m":      [np.median(h3_dists), np.median(s2_dists)],
    "random_mean_m": [random_mean_m(h3_sorted, h3_dist_m), random_mean_m(s2_sorted, s2_dist_m)],
    "share_near":    [(h3_dists <= NEAR_M).mean(), (s2_dists <= NEAR_M).mean()],
    "jumps_gt_3km":  [(h3_dists > 3000).sum(), (s2_dists > 3000).sum()],
}, index=["H3 res8 (center)", "S2 L13 (clip)"])
order["mean_in_steps"] = order["mean_m"] / order["step_m"]
order["x_better_than_random"] = order["random_mean_m"] / order["mean_m"]
display(order.round(2))

# H3 не «без порядка»: заметно лучше случайного, но соседей по id у S2 больше
assert order.loc["H3 res8 (center)", "x_better_than_random"] > 2
assert order.loc["S2 L13 (clip)", "share_near"] > order.loc["H3 res8 (center)", "share_near"]

**Вывод (задание 1).** Подряд идущие по id ячейки оказываются соседями (≤1.5 км) у **S2 в 90% пар, у H3 в 44%**.
Утверждение «у H3 порядка нет» не подтвердилось: H3 в ~3.7 раза лучше случайного порядка (S2 в ~6.4 раза), потому что
дети одного родителя идут подряд по id. Но на границах родителей id прыгают (17 скачков >3 км против 9 у S2), а кривая
Гильберта почти всегда даёт следующей соседнюю ячейку.

In [ ]:
# Задание 2

## Шаг 6 — карта

In [ ]:
import folium
center = (POLY.centroid.y, POLY.centroid.x)
m = folium.Map(location=center, zoom_start=12, tiles="OpenStreetMap")   # cartodbpositron теперь требует API-ключ
folium.GeoJson(mapping(POLY), name="полигон",
               style_function=lambda _: {"color": "#111111", "weight": 2, "fill": False}).add_to(m)

fg = folium.FeatureGroup(name="H3 res 8 (center)", show=True)
for c in h3_cells[(8, "center")]:
    folium.Polygon(h3.cell_to_boundary(c), color="#1F5E96", weight=1,
                   fill=True, fill_opacity=0.12, tooltip=f"H3 {c}").add_to(fg)
fg.add_to(m)

fg2 = folium.FeatureGroup(name="S2 L10-14 (max_cells=500)", show=True)
for i in s2_cells[("s2", 500)]:
    ring = [(y, x) for x, y in s2_cell_polygon(i).exterior.coords]
    folium.Polygon(ring, color="#C4561D", weight=1, fill=True, fill_opacity=0.10,
                   tooltip=f"S2 L{i.level()} {i.to_token()}").add_to(fg2)
fg2.add_to(m)

folium.LayerControl().add_to(m)
m.save("m01/s2_vs_h3.html"); m